# Experimental diffusion notebook: LoRA training

Status: `experimental-diffusion`, not the stable-core API path.

Current implemented scope:
- minimal `SD1.5 LoRA` training recipe
- folder dataset contract (`image + .txt` or `metadata.jsonl` / `metadata.csv`)
- `UNet LoRA` by default; `text_encoder LoRA` is optional

Environment assumptions:
- `torch` + training dependencies installed
- usually CUDA/GPU and enough VRAM
- local dataset preparation is required

Notebook policy: treat outputs as illustrative and re-check against `documentation/API_SURFACE.md` and `documentation/DIFFUSION_STATUS.md`.

# Обучение в YggDrasill

### Обучение LoRA модели

In [ ]:
# Как обучать LoRA модель на YggDrasill

# Текущая реализация в репозитории поддерживает **минимальный training path для `SD1.5 LoRA`** через `yggdrasill.integrations.diffusers.training`.

# Что можно использовать как датасет:
# - локальную папку с изображениями;
# - для каждого изображения подпись в соседнем `.txt` файле,
#   или общий `metadata.jsonl` / `metadata.csv`;
# - Hugging Face dataset, если в `data_dir` передать dataset id.

# Для Hugging Face dataset можно дополнительно задать:
# - `dataset_split`, например `train`;
# - `dataset_config_name`, если у датасета есть несколько конфигураций.

# Нужное окружение:
# - `torch`, `diffusers`, `datasets`, `peft`, `torchvision`, `Pillow`.

# Базовый сценарий обучения:
# - вызвать `train_sd15_lora(...)`;
# - передать `data_dir`, `pretrained`, `output_path`;
# - выбрать `resolution`, `lora_rank`, `lora_alpha`, `lr`, `num_epochs`;
# - по умолчанию обучается `UNet LoRA`, а `train_text_encoder=True` включает также LoRA для text encoder.

# Примеры ниже показывают оба варианта: локальный folder dataset и Hugging Face dataset.

In [ ]:
from yggdrasill.integrations.diffusers.training import train_sd15_lora

# Local dataset contract:
# data/
#   image_001.png
#   image_001.txt
#   image_002.png
#   image_002.txt
# or a metadata.jsonl / metadata.csv manifest in the same folder.

result = train_sd15_lora(
    data_dir="GrafikXxxxxxxYyyyyyyyyyy/Achoocation",
    pretrained="runwayml/stable-diffusion-v1-5",
    output_path="artifacts/my_sd15_lora.safetensors",
    mixed_precision="fp16",
    resolution=512,
    batch_size=5,
    gradient_accumulation_steps=1,
    lr=1e-4,
    num_epochs=1000,
    max_train_steps=10000,
    lora_rank=64,
    lora_alpha=64,
    train_vae=False,
    train_text_encoder=True,
    checkpoint_every_n_steps=250,
    logging_steps=10,
)

result

### Проверка обученной LoRA

In [ ]:
from yggdrasill import DiffusionGraphBuilder

builder = DiffusionGraphBuilder.from_template("sd15_img2img", device="cuda")
builder.add_component("lora", "sd15.lora", pretrained="artifacts/my_sd15_lora.safetensors")

img = builder.run(
    prompt="Achoocation XJV523 is standing near a fence",
    image="https://sun9-71.userapi.com/s/v1/ig2/WzU87FEf42ixJMUe0jGTZLfE4LsP2myo5HiLsMjp6hr-RAAkhra6d7YH5ivHZSSsvONTrZlClRP8ykmnrRxY6LOR.jpg?quality=95&as=32x24,48x36,72x54,108x81,160x120,240x180,360x270,480x360,540x405,640x480,720x540,1080x810,1280x960,1440x1080,2560x1920&from=bu&u=Ejz89v6iTxvC-lRbNEuA92-HuEbr_sIL7_gob1OfnZw&cs=2560x0",
    strength=0.9,
    negative_prompt="blurry",
    lora_conditioning_scale=0.7,
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
).images[0]

img